# STRIVE Final Advanced Analysis From the Paper Artifact

This notebook loads the merged paper artifact ZIP and reproduces the advanced tables and
figures without trajectory generation, sandbox execution, PRM loading, or API calls.

The artifact already contains the authoritative repaired metrics. GPT-OSS remains in the
operational coverage/correctness/latency tables, but is excluded from trace-derived
analyses (`G`, `V`, `Q`, `PTU`, redundancy, token budgets, and Pareto trace comparisons)
because its provider reasoning was not reliably mapped to visible protocol actions.


## 1. Setup


In [1]:
%pip install -q pandas numpy matplotlib seaborn scipy



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import math
import shutil
import zipfile
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import binomtest, spearmanr

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
RNG_SEED = 42

try:
    from IPython.display import display
except Exception:
    display = print


## 2. Load one metrics ZIP


In [3]:
RESULT_INPUT = (
    "/path/to/your/project/"
    "i-got-these-reviews-from-a/paper_artifact_build/"
    "strive_paper_artifact_20260715T160237Z.zip"
)
RUN_LABEL = "strive_v11_final_paper_artifact"
OUTPUT_DIR = Path(
    "/kaggle/working/strive_final_advanced_analysis"
    if Path("/kaggle/working").exists()
    else Path.cwd() / "strive_final_advanced_analysis"
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_SAMPLES = 5000
TOKEN_BUDGETS = [512, 1024, 2048, 4096, 8192]
REDUNDANCY_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
INFRASTRUCTURE_STOPS = {"api_error", "worker_exception", "trajectory_timeout"}
TRACE_EXCLUDED_AGENTS = {"gpt-oss-20b"}


In [4]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def safe_extract_zip(path, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(target)


def resolve_bundle(value):
    source = Path(value).expanduser().resolve()
    if not source.exists():
        raise FileNotFoundError(source)
    if source.is_file():
        if source.suffix.lower() != ".zip":
            raise ValueError("RESULT_INPUT must be a ZIP or directory")
        target = OUTPUT_DIR / "loaded_bundle" / source.stem
        if target.exists():
            shutil.rmtree(target)
        safe_extract_zip(source, target)
        source = target
    if (source / "manifest.json").is_file():
        return source
    manifests = list(source.rglob("manifest.json"))
    if len(manifests) != 1:
        raise RuntimeError(f"Expected one manifest.json under {source}, found {len(manifests)}")
    return manifests[0].parent


def first_existing(root, *relative_paths):
    for relative in relative_paths:
        candidate = root / relative
        if candidate.exists():
            return candidate
    return None


def load_bundle(value):
    root = resolve_bundle(value)
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    metrics_path = first_existing(root, "metrics.jsonl", "raw_data/metrics.jsonl")
    trajectories_path = first_existing(root, "trajectories.jsonl", "raw_data/trajectories.jsonl")
    problem_path = first_existing(root, "problems.jsonl", "raw_data/problems.jsonl")
    prm_path = first_existing(root, "prm_scores.json", "raw_data/prm_scores.json")
    if metrics_path is None or trajectories_path is None:
        raise RuntimeError("Artifact is missing authoritative metrics or trajectories")
    return {
        "root": root,
        "manifest": manifest,
        "metrics_raw": read_jsonl(metrics_path),
        "trajectories_raw": read_jsonl(trajectories_path),
        "problems_raw": read_jsonl(problem_path) if problem_path else [],
        "prm_scores": json.loads(prm_path.read_text()) if prm_path else {},
    }


if not RESULT_INPUT:
    raise RuntimeError("Set RESULT_INPUT to the uploaded paper artifact ZIP before continuing.")
bundle = load_bundle(RESULT_INPUT)
metrics_raw_all = bundle["metrics_raw"]
trajectories_raw = bundle["trajectories_raw"]
prm_scores_all = bundle["prm_scores"]
manifest = bundle["manifest"]
trace_agents = sorted({
    row.get("agent") for row in metrics_raw_all
    if row.get("agent") not in TRACE_EXCLUDED_AGENTS
})
metrics_raw = [row for row in metrics_raw_all if row.get("agent") in trace_agents]
prm_scores = {agent: value for agent, value in prm_scores_all.items() if agent in trace_agents}
print("Loaded all metric rows:", len(metrics_raw_all))
print("Loaded trace-comparable rows:", len(metrics_raw))
print("Loaded trajectory rows:", len(trajectories_raw))
print("Operational agents:", sorted({row.get("agent") for row in metrics_raw_all}))
print("Trace-comparable agents:", trace_agents)
print("Run:", manifest.get("run_name", manifest.get("artifact_type")))


Loaded all metric rows: 1800
Loaded trace-comparable rows: 1500
Loaded trajectory rows: 1800
Operational agents: ['glm-5.2', 'gpt-5-nano', 'gpt-oss-20b', 'minimax-m3', 'ministral-14b', 'nemotron-3-nano']
Trace-comparable agents: ['glm-5.2', 'gpt-5-nano', 'minimax-m3', 'ministral-14b', 'nemotron-3-nano']
Run: strive_paper_metrics_and_figures


## 3. Canonical analysis tables


In [5]:
metrics_df = pd.json_normalize(metrics_raw, sep=".")
operational_df = pd.json_normalize(metrics_raw_all, sep=".")
trajectories_df = pd.json_normalize(trajectories_raw, sep=".")
trajectories_by_key = {
    (row["agent_name"], row["problem_id"]): row for row in trajectories_raw
}
if len(trajectories_by_key) != len(trajectories_raw):
    raise RuntimeError("Duplicate (agent_name, problem_id) trajectory records detected")

COLUMN_MAP = {
    "C": "C_G_V.C_final", "G": "C_G_V.G", "V": "C_G_V.V",
    "G_level": "C_G_V.G_level", "correctness_source": "C_G_V.correctness_source",
    "judge_used": "C_G_V.judge_used", "needs_judge": "C_G_V.needs_judge",
    "Q": "Q.Q_step", "PTU": "T.PTU", "billable": "T.T_billable",
    "useful": "T.T_useful", "pre_waste": "T.T_pre_waste",
    "post_waste": "T.T_post_waste", "answer_reporting": "T.T_answer_reporting",
    "hidden": "T.T_hidden_reasoning", "solution_step": "T.solution_evidence_step",
    "R_harmful": "R.R_harmful", "R_useful": "R.R_useful_verification",
    "R_semantic": "R.R_semantic_raw", "latency": "L.L_trajectory_sec",
    "service": "L.L_provider_service_sec", "pace": "L.L_rate_limit_wait_sec",
    "retry_wait": "L.L_retry_wait_sec", "retry_count": "L.retry_count",
}
for frame in (metrics_df, operational_df):
    for short, long_name in COLUMN_MAP.items():
        frame[short] = frame[long_name] if long_name in frame else np.nan
    for required in ("agent", "problem_id", "dataset", "subject", "finished", "stop_reason"):
        if required not in frame:
            frame[required] = np.nan
    frame["infrastructure_failure"] = frame["stop_reason"].isin(INFRASTRUCTURE_STOPS)
    frame["completed"] = frame["finished"].fillna(False).astype(bool)
    frame["level"] = [
        trajectories_by_key[(row.agent, row.problem_id)].get("level", "unknown")
        for row in frame[["agent", "problem_id"]].itertuples(index=False)
    ]
core_metrics = ["C", "G", "V", "Q", "PTU", "R_harmful"]

# Preserve the authoritative final tables generated from the two-bundle merge.
final_metrics_dir = bundle["root"] / "final_metrics"
final_core_all = pd.read_csv(final_metrics_dir / "core_metrics_all_models.csv") if (final_metrics_dir / "core_metrics_all_models.csv").exists() else pd.DataFrame()
final_core_trace = pd.read_csv(final_metrics_dir / "core_metrics_trace_comparable.csv") if (final_metrics_dir / "core_metrics_trace_comparable.csv").exists() else pd.DataFrame()
final_core_all.to_csv(OUTPUT_DIR / "final_core_metrics_all_models.csv", index=False)
final_core_trace.to_csv(OUTPUT_DIR / "final_core_metrics_trace_comparable.csv", index=False)
display(final_core_all.round(4))


,agent,attempted_n,valid_n,coverage,format_adherence,infrastructure_failure_rate,C_operational,C_valid,G_operational,G_valid,V_operational,V_valid,Q,PTU,R_harmful,E_optional,total_billable_tokens,avg_billable_tokens_per_attempt,tokens_per_correct_solve,tokens_per_verified_solve,avg_latency_sec,p95_latency_sec,trace_metric_eligible,trace_exclusion_reason
0,glm-5.2,300,300,1.0000,1.0000,0.0000,0.7300,0.7300,0.6200,0.6200,0.5633,0.5633,0.4862,0.2550,0.0114,0.1884,561621.0,1872.0700,2564.4795,3323.2012,49.2026,125.9392,True,NaN
1,gpt-5-nano,300,280,0.9333,0.9633,0.0300,0.6667,0.7143,0.7267,0.7786,0.6100,0.6536,0.4705,0.2798,0.1357,0.1207,972031.0,3240.1033,4860.1550,5311.6448,10.1239,23.2433,True,NaN
2,gpt-oss-20b,300,260,0.8667,0.8700,0.0033,0.5967,0.6885,0.0167,0.0192,0.0100,0.0115,NaN,NaN,NaN,NaN,327149.0,1090.4967,1827.6480,109049.6667,8.7938,19.4975,False,provider reasoning is not reliably mapped to v...
3,minimax-m3,300,298,0.9933,0.9967,0.0033,0.7600,0.7651,0.4733,0.4765,0.3900,0.3926,0.4485,0.2715,0.0041,0.1731,1029767.0,3432.5567,4516.5219,8801.4274,113.6326,341.8315,True,NaN
4,ministral-14b,300,296,0.9867,1.0000,0.0133,0.3633,0.3682,0.1900,0.1926,0.1733,0.1757,0.1643,0.0998,0.0116,0.0828,494148.0,1647.1600,4533.4679,9502.8462,26.4038,93.0781,True,NaN
5,nemotron-3-nano,300,299,0.9967,0.9967,0.0000,0.6167,0.6187,0.7400,0.7425,0.5500,0.5518,0.4161,0.2392,0.1929,0.0861,1216905.0,4056.3500,6577.8649,7375.1818,19.2348,52.0563,True,NaN


## 4. Coverage, missingness, and clean-run sensitivity


In [6]:
coverage_rows = []
for agent, group in operational_df.groupby("agent"):
    row = {
        "agent": agent,
        "n": len(group),
        "unique_problems": group["problem_id"].nunique(),
        "completion_rate": group["completed"].mean(),
        "infrastructure_failure_rate": group["infrastructure_failure"].mean(),
        "judge_fallbacks": int(group["judge_used"].fillna(False).sum()),
        "unresolved_judgments": int(group["needs_judge"].fillna(False).sum()),
        "trace_metric_eligible": agent in trace_agents,
    }
    for metric in core_metrics:
        # Coverage is always counted within this agent's operational rows.
        # Trace eligibility only controls whether trace-derived fields are
        # reported, not the denominator used for per-agent missingness.
        analysis_frame = group
        if agent not in trace_agents and metric in {"Q", "PTU", "R_harmful"}:
            row[f"{metric}_nan"] = len(group)
            row[f"{metric}_zero"] = np.nan
        else:
            row[f"{metric}_nan"] = int(analysis_frame[metric].isna().sum())
            row[f"{metric}_zero"] = int((analysis_frame[metric] == 0).sum())
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows).sort_values("agent")
display(coverage)
coverage.to_csv(OUTPUT_DIR / "coverage_and_missingness.csv", index=False)

# Operational sensitivity retains all six models for C/G/V; trace scores are shown
# only for the five eligible models in the other advanced sections.
sensitivity = pd.concat([
    frame.groupby("agent")[["C", "G", "V"]].mean().assign(view=name).reset_index()
    for name, frame in {
        "all_attempted": operational_df,
        "completed_only": operational_df[operational_df["completed"]],
        "infrastructure_clean": operational_df[~operational_df["infrastructure_failure"]],
    }.items()
], ignore_index=True)
display(sensitivity)
sensitivity.to_csv(OUTPUT_DIR / "clean_run_sensitivity.csv", index=False)
g = sns.catplot(
    data=sensitivity.melt(id_vars=["view", "agent"], value_vars=["C", "G", "V"]),
    x="agent", y="value", hue="variable", col="view", kind="bar",
    height=4.2, aspect=1.15, sharex=False,
)
g.set_xticklabels(rotation=30)
g.set(ylim=(0, 1))
g.fig.savefig(OUTPUT_DIR / "clean_run_sensitivity.png", dpi=180, bbox_inches="tight")
plt.show()


,agent,n,unique_problems,completion_rate,infrastructure_failure_rate,judge_fallbacks,unresolved_judgments,trace_metric_eligible,C_nan,C_zero,G_nan,G_zero,V_nan,V_zero,Q_nan,Q_zero,PTU_nan,PTU_zero,R_harmful_nan,R_harmful_zero
0,glm-5.2,300,300,1.000000,0.000000,21,0,True,0,81,0,114,0,131,0,78.0,0,204.0,0,287.0
1,gpt-5-nano,300,300,0.933333,0.030000,21,0,True,0,100,0,82,0,117,20,30.0,20,110.0,20,153.0
2,gpt-oss-20b,300,300,0.866667,0.003333,33,0,False,0,121,0,295,0,297,300,NaN,300,NaN,300,NaN
3,minimax-m3,300,300,0.993333,0.003333,20,0,True,0,72,0,158,0,183,2,93.0,2,190.0,2,293.0
4,ministral-14b,300,300,0.986667,0.013333,19,0,True,0,191,0,243,0,248,4,216.0,4,258.0,4,285.0
5,nemotron-3-nano,300,300,0.996667,0.000000,20,0,True,0,115,0,78,0,135,1,35.0,1,140.0,1,112.0


,agent,C,G,V,view
0,glm-5.2,0.730000,0.620000,0.563333,all_attempted
1,gpt-5-nano,0.666667,0.726667,0.610000,all_attempted
2,gpt-oss-20b,0.596667,0.016667,0.010000,all_attempted
3,minimax-m3,0.760000,0.473333,0.390000,all_attempted
4,ministral-14b,0.363333,0.190000,0.173333,all_attempted
5,nemotron-3-nano,0.616667,0.740000,0.550000,all_attempted
6,glm-5.2,0.730000,0.620000,0.563333,completed_only
7,gpt-5-nano,0.714286,0.778571,0.653571,completed_only
8,gpt-oss-20b,0.688462,0.019231,0.011538,completed_only
9,minimax-m3,0.765101,0.476510,0.392617,completed_only


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/2658998180.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Correctness-grounding matrix and grounding levels


In [7]:
cg = (
    metrics_df.assign(
        correctness=np.where(metrics_df["C"] == 1, "correct", "incorrect"),
        grounding=np.where(metrics_df["G"] == 1, "grounded", "ungrounded"),
    )
    .groupby(["agent", "correctness", "grounding"]).size()
    .rename("count").reset_index()
)
cg["rate"] = cg["count"] / cg.groupby("agent")["count"].transform("sum")
display(cg)
cg.to_csv(OUTPUT_DIR / "correctness_grounding_matrix.csv", index=False)

retention = metrics_df.groupby("agent").agg(C=("C", "mean"), G=("G", "mean"), V=("V", "mean"))
retention["VerificationGap"] = retention["C"] - retention["V"]
retention["GroundingRetention"] = retention["V"] / retention["C"].replace(0, np.nan)
display(retention.sort_values("V", ascending=False))
retention.to_csv(OUTPUT_DIR / "verification_gap.csv")

g_levels = metrics_df.groupby(["agent", "G_level"]).size().rename("count").reset_index()
g_levels["rate"] = g_levels["count"] / g_levels.groupby("agent")["count"].transform("sum")
pivot_levels = g_levels.pivot(index="agent", columns="G_level", values="rate").fillna(0)
pivot_levels.plot(kind="bar", stacked=True, figsize=(10, 5), ylim=(0, 1), title="Grounding levels")
plt.ylabel("Trajectory rate")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grounding_levels.png", dpi=180, bbox_inches="tight")
plt.show()

adjudication = (
    metrics_df.groupby(["agent", "correctness_source"]).size()
    .rename("count").reset_index()
)
adjudication["rate"] = adjudication["count"] / adjudication.groupby("agent")["count"].transform("sum")
display(adjudication)
adjudication.to_csv(OUTPUT_DIR / "correctness_adjudication_sources.csv", index=False)


,agent,correctness,grounding,count,rate
0,glm-5.2,correct,grounded,169,0.563333
1,glm-5.2,correct,ungrounded,50,0.166667
2,glm-5.2,incorrect,grounded,17,0.056667
3,glm-5.2,incorrect,ungrounded,64,0.213333
4,gpt-5-nano,correct,grounded,183,0.610000
5,gpt-5-nano,correct,ungrounded,17,0.056667
6,gpt-5-nano,incorrect,grounded,35,0.116667
7,gpt-5-nano,incorrect,ungrounded,65,0.216667
8,minimax-m3,correct,grounded,117,0.390000
9,minimax-m3,correct,ungrounded,111,0.370000


,C,G,V,VerificationGap,GroundingRetention
agent,,,,,
gpt-5-nano,0.666667,0.726667,0.610000,0.056667,0.915000
glm-5.2,0.730000,0.620000,0.563333,0.166667,0.771689
nemotron-3-nano,0.616667,0.740000,0.550000,0.066667,0.891892
minimax-m3,0.760000,0.473333,0.390000,0.370000,0.513158
ministral-14b,0.363333,0.190000,0.173333,0.190000,0.477064


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/3965658457.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,agent,correctness_source,count,rate
0,glm-5.2,judge_fallback,21,0.070000
1,glm-5.2,symbolic,279,0.930000
2,gpt-5-nano,generation_failure,20,0.066667
3,gpt-5-nano,judge_fallback,21,0.070000
4,gpt-5-nano,symbolic,259,0.863333
5,minimax-m3,generation_failure,2,0.006667
6,minimax-m3,judge_fallback,20,0.066667
7,minimax-m3,symbolic,278,0.926667
8,ministral-14b,generation_failure,4,0.013333
9,ministral-14b,judge_fallback,19,0.063333


## 6. Bootstrap confidence intervals and paired tests


In [8]:
def bootstrap_mean_ci(values, samples=BOOTSTRAP_SAMPLES, seed=RNG_SEED):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(samples, len(values)), replace=True).mean(axis=1)
    return float(values.mean()), *np.quantile(draws, [0.025, 0.975]).tolist()


ci_rows = []
for agent, group in metrics_df.groupby("agent"):
    for metric in core_metrics:
        mean, low, high = bootstrap_mean_ci(group[metric])
        ci_rows.append({"agent": agent, "metric": metric, "mean": mean, "low": low, "high": high})
metric_ci = pd.DataFrame(ci_rows)
display(metric_ci)
metric_ci.to_csv(OUTPUT_DIR / "metric_bootstrap_ci.csv", index=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, metric in zip(axes.flat, core_metrics):
    part = metric_ci[metric_ci["metric"] == metric].sort_values("mean")
    ax.errorbar(
        part["mean"], part["agent"],
        xerr=[part["mean"] - part["low"], part["high"] - part["mean"]],
        fmt="o", capsize=3,
    )
    ax.set_title(metric)
    ax.set_xlim(0, 1)
fig.suptitle("Problem-level bootstrap 95% confidence intervals")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "metric_ci_forest.png", dpi=180, bbox_inches="tight")
plt.show()

def holm_adjust(p_values):
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running = 0.0
    m = len(p_values)
    for rank, index in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p_values[index]))
        adjusted[index] = running
    return adjusted


paired_rows = []
for metric in ("C", "V"):
    for left, right in combinations(sorted(metrics_df["agent"].unique()), 2):
        a = metrics_df[metrics_df["agent"] == left][["problem_id", metric]]
        b = metrics_df[metrics_df["agent"] == right][["problem_id", metric]]
        pair = a.merge(b, on="problem_id", suffixes=("_left", "_right")).dropna()
        x, y = pair[f"{metric}_left"].to_numpy(), pair[f"{metric}_right"].to_numpy()
        discordant_left = int(((x == 1) & (y == 0)).sum())
        discordant_right = int(((x == 0) & (y == 1)).sum())
        discordant = discordant_left + discordant_right
        p = binomtest(discordant_left, discordant, 0.5).pvalue if discordant else 1.0
        rng = np.random.default_rng(RNG_SEED)
        differences = x - y
        draws = rng.choice(differences, size=(BOOTSTRAP_SAMPLES, len(differences)), replace=True).mean(axis=1)
        paired_rows.append({
            "metric": metric, "left": left, "right": right, "n": len(pair),
            "mean_difference": differences.mean(),
            "ci_low": np.quantile(draws, 0.025), "ci_high": np.quantile(draws, 0.975),
            "mcnemar_exact_p": p,
        })
paired_tests = pd.DataFrame(paired_rows)
paired_tests["holm_p"] = holm_adjust(paired_tests["mcnemar_exact_p"].to_numpy())
display(paired_tests.sort_values(["metric", "holm_p"]))
paired_tests.to_csv(OUTPUT_DIR / "paired_model_tests.csv", index=False)


,agent,metric,mean,low,high
0,glm-5.2,C,0.730000,0.680000,0.780000
1,glm-5.2,G,0.620000,0.566583,0.673333
2,glm-5.2,V,0.563333,0.506667,0.620000
3,glm-5.2,Q,0.486250,0.446944,0.524917
4,glm-5.2,PTU,0.255002,0.209181,0.301274
5,glm-5.2,R_harmful,0.011444,0.005500,0.018222
6,gpt-5-nano,C,0.666667,0.613333,0.720000
7,gpt-5-nano,G,0.726667,0.676667,0.776667
8,gpt-5-nano,V,0.610000,0.553333,0.666667
9,gpt-5-nano,Q,0.470536,0.442409,0.497918


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1248210192.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,metric,left,right,n,mean_difference,ci_low,ci_high,mcnemar_exact_p,holm_p
7,C,minimax-m3,ministral-14b,300,0.396667,0.330000,0.460000,4.417856e-28,8.393926e-27
2,C,glm-5.2,ministral-14b,300,0.366667,0.303333,0.430000,1.396050e-24,2.373285e-23
5,C,gpt-5-nano,ministral-14b,300,0.303333,0.236667,0.370000,4.569611e-17,6.854416e-16
9,C,ministral-14b,nemotron-3-nano,300,-0.253333,-0.326667,-0.183333,2.619697e-11,3.667576e-10
8,C,minimax-m3,nemotron-3-nano,300,0.143333,0.080000,0.206667,1.476695e-05,1.476695e-04
3,C,glm-5.2,nemotron-3-nano,300,0.113333,0.053333,0.173333,3.169993e-04,2.535994e-03
4,C,gpt-5-nano,minimax-m3,300,-0.093333,-0.150000,-0.036667,2.031140e-03,1.421798e-02
0,C,glm-5.2,gpt-5-nano,300,0.063333,0.009917,0.116667,3.192716e-02,1.915630e-01
6,C,gpt-5-nano,nemotron-3-nano,300,0.050000,-0.006667,0.110000,1.192737e-01,4.770947e-01
1,C,glm-5.2,minimax-m3,300,-0.030000,-0.080000,0.020000,3.056774e-01,6.113548e-01


## 7. Dataset, subject, and difficulty breakdowns


In [9]:
breakdown = metrics_df.groupby(["agent", "dataset", "subject"])[core_metrics].agg(["mean", "count"])
display(breakdown)
breakdown.to_csv(OUTPUT_DIR / "dataset_subject_breakdown.csv")

subject_v = metrics_df.pivot_table(index="subject", columns="agent", values="V", aggfunc="mean")
plt.figure(figsize=(11, max(4, 0.45 * len(subject_v))))
sns.heatmap(subject_v, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title("Verified solve rate by subject")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "subject_verified_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

dataset_metrics = metrics_df.groupby(["dataset", "agent"])[["C", "G", "V", "Q", "PTU"]].mean().reset_index()
display(dataset_metrics)
dataset_metrics.to_csv(OUTPUT_DIR / "dataset_metrics.csv", index=False)

difficulty_metrics = metrics_df.groupby(["level", "agent"])[["C", "G", "V", "Q", "PTU"]].agg(["mean", "count"])
display(difficulty_metrics)
difficulty_metrics.to_csv(OUTPUT_DIR / "difficulty_metrics.csv")


C               G               V               Q             PTU       R_harmful      
                                                                              mean count      mean count      mean count      mean count      mean count      mean count
agent           dataset                           subject                                                                                                               
glm-5.2         MATH-500                          Algebra                 0.870968    31  0.774194    31  0.709677    31  0.629032    31  0.402899    31  0.008065    31
                                                  Counting & Probability  0.916667    24  0.958333    24  0.916667    24  0.682292    24  0.416667    24  0.000000    24
                                                  Geometry                0.555556    27  0.407407    27  0.333333    27  0.274691    27  0.037037    27  0.000000    27
                                                  Intermediate Algebra    0.833333    30  0.700000    30  0.633333    30  0.515556    30  0.157674    30  0.000000    30
                                                  Number Theory           0.933333    30  0.900000    30  0.866667    30  0.708333    30  0.448924    30  0.011111    30
                                                  Prealgebra              0.933333    30  0.766667    30  0.766667    30  0.697222    30  0.483224    30  0.000000    30
                                                  Precalculus             0.678571    28  0.321429    28  0.321429    28  0.467262    28  0.341935    28  0.000000    28
                OlympiadBench-OE-TO-maths-en-COMP Algebra                 0.680000    25  0.560000    25  0.440000    25  0.402333    25  0.159162    25  0.023333    25
                                                  Combinatorics           0.400000    25  0.360000    25  0.240000    25  0.201000    25  0.085522    25  0.045333    25
                                                  Geometry                0.440000    25  0.280000    25  0.200000    25  0.266000    25  0.118704    25  0.024000    25
                                                  Number Theory           0.680000    25  0.720000    25  0.680000    25  0.405333    25  0.066287    25  0.021333    25
gpt-5-nano      MATH-500                          Algebra                 0.870968    31  0.903226    31  0.838710    31  0.499059    31  0.337601    31  0.188172    31
                                                  Counting & Probability  0.875000    24  0.875000    24  0.833333    24  0.549638    23  0.313247    23  0.115217    23
                                                  Geometry                0.592593    27  0.666667    27  0.555556    27  0.459091    22  0.332922    22  0.181818    22
                                                  Intermediate Algebra    0.666667    30  0.633333    30  0.600000    30  0.426006    29  0.201806    29  0.102299    29
                                                  Number Theory           0.833333    30  0.866667    30  0.800000    30  0.531034    29  0.242996    29  0.110345    29
                                                  Prealgebra              0.866667    30  0.800000    30  0.766667    30  0.536944    30  0.375737    30  0.163889    30
                                                  Precalculus             0.535714    28  0.571429    28  0.357143    28  0.449000    25  0.282998    25  0.160000    25
                OlympiadBench-OE-TO-maths-en-COMP Algebra                 0.560000    25  0.760000    25  0.560000    25  0.530159    21  0.379485    21  0.104762    21
                                                  Combinatorics           0.440000    25  0.720000    25  0.400000    25  0.396591    22  0.179140    22  0.147727    22
                                                  Geometry                0.400000    25  0.520000    25  0.320000    25  0.367014    24  0.263713    24  0.090972    24
                         

/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/3486060170.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,dataset,agent,C,G,V,Q,PTU
0,MATH-500,glm-5.2,0.820,0.690,0.650,0.570042,0.328793
1,MATH-500,gpt-5-nano,0.750,0.760,0.680,0.493651,0.297571
2,MATH-500,minimax-m3,0.865,0.455,0.430,0.478917,0.307779
3,MATH-500,ministral-14b,0.460,0.240,0.220,0.216018,0.143168
4,MATH-500,nemotron-3-nano,0.685,0.765,0.615,0.424208,0.261513
5,OlympiadBench-OE-TO-maths-en-COMP,glm-5.2,0.550,0.480,0.390,0.318667,0.107419
6,OlympiadBench-OE-TO-maths-en-COMP,gpt-5-nano,0.500,0.660,0.470,0.422527,0.242936
7,OlympiadBench-OE-TO-maths-en-COMP,minimax-m3,0.550,0.510,0.310,0.386352,0.197429
8,OlympiadBench-OE-TO-maths-en-COMP,ministral-14b,0.170,0.090,0.080,0.058204,0.010778
9,OlympiadBench-OE-TO-maths-en-COMP,nemotron-3-nano,0.480,0.690,0.420,0.399790,0.194243


C               G               V               Q             PTU      
                                 mean count      mean count      mean count      mean count      mean count
level       agent                                                                                          
1           glm-5.2          0.900000    30  0.766667    30  0.733333    30  0.813889    30  0.773000    30
            gpt-5-nano       0.800000    30  0.766667    30  0.733333    30  0.584077    28  0.434508    28
            minimax-m3       0.900000    30  0.466667    30  0.466667    30  0.536111    30  0.479512    30
            ministral-14b    0.766667    30  0.466667    30  0.466667    30  0.459722    30  0.385116    30
            nemotron-3-nano  0.666667    30  0.766667    30  0.600000    30  0.411944    30  0.248045    30
2           glm-5.2          0.883721    43  0.697674    43  0.674419    43  0.684109    43  0.519660    43
            gpt-5-nano       0.883721    43  0.813953    43  0.790698    43  0.493552    42  0.312388    42
            minimax-m3       0.930233    43  0.395349    43  0.395349    43  0.500388    43  0.296074    43
            ministral-14b    0.674419    43  0.372093    43  0.372093    43  0.349903    43  0.232558    43
            nemotron-3-nano  0.813953    43  0.744186    43  0.697674    43  0.455329    43  0.358937    43
3           glm-5.2          0.850000    40  0.700000    40  0.650000    40  0.518750    40  0.200000    40
            gpt-5-nano       0.750000    40  0.775000    40  0.700000    40  0.469189    38  0.264112    38
            minimax-m3       0.825000    40  0.425000    40  0.350000    40  0.463750    40  0.321685    40
            ministral-14b    0.475000    40  0.250000    40  0.225000    40  0.170417    40  0.115823    40
            nemotron-3-nano  0.775000    40  0.850000    40  0.725000    40  0.486354    40  0.337200    40
4           glm-5.2          0.738095    42  0.642857    42  0.571429    42  0.443651    42  0.173423    42
            gpt-5-nano       0.690476    42  0.714286    42  0.619048    42  0.483446    37  0.221734    37
            minimax-m3       0.809524    42  0.452381    42  0.428571    42  0.501984    42  0.253799    42
            ministral-14b    0.333333    42  0.119048    42  0.095238    42  0.108537    41  0.030975    41
            nemotron-3-nano  0.595238    42  0.690476    42  0.523810    42  0.379266    42  0.133942    42
5           glm-5.2          0.755556    45  0.666667    45  0.644444    45  0.462037    45  0.109768    45
            gpt-5-nano       0.644444    45  0.733333    45  0.577778    45  0.465909    44  0.288954    44
            minimax-m3       0.866667    45  0.533333    45  0.511111    45  0.412222    45  0.242496    45
            ministral-14b    0.155556    45  0.066667    45  0.022222    45  0.064074    45  0.022980    45
            nemotron-3-nano  0.577778    45  0.777778    45  0.533333    45  0.389352    45  0.229185    45
Competition glm-5.2          0.550000   100  0.480000   100  0.390000   100  0.318667   100  0.107419   100
            gpt-5-nano       0.500000   100  0.660000   100  0.470000   100  0.422527    91  0.242936    91
            minimax-m3       0.550000   100  0.510000   100  0.310000   100  0.386352    98  0.197429    98
            ministral-14b    0.170000   100  0.090000   100  0.080000   100  0.058204    97  0.010778    97
            nemotron-3-nano  0.480000   100  0.690000   100  0.420000   100  0.399790    99  0.194243    99

## 8. Cost, token composition, and budget curves


In [10]:
cost_rows = []
for agent, group in metrics_df.groupby("agent"):
    correct = group["C"].sum()
    verified = group["V"].sum()
    billable = group["billable"].sum()
    latency = group["latency"].sum()
    cost_rows.append({
        "agent": agent,
        "avg_billable_tokens": group["billable"].mean(),
        "tokens_per_correct_solve": billable / correct if correct else np.nan,
        "tokens_per_verified_solve": billable / verified if verified else np.nan,
        "seconds_per_correct_solve": latency / correct if correct else np.nan,
        "seconds_per_verified_solve": latency / verified if verified else np.nan,
        "hidden_reasoning_share": group["hidden"].sum() / max(group["T.T_api_output"].sum(), 1),
        "answer_reporting_share": group["answer_reporting"].sum() / max(group["T.T_api_output"].sum(), 1),
    })
cost_table = pd.DataFrame(cost_rows).sort_values("tokens_per_verified_solve")
display(cost_table)
cost_table.to_csv(OUTPUT_DIR / "cost_per_solve.csv", index=False)

def budget_status(metric_row, trajectory, budget):
    cumulative = 0
    answer_visible = False
    evidence_visible = False
    evidence_step = metric_row.get("C_G_V.grounding_evidence_step")
    for index, step in enumerate(trajectory.get("steps", []), start=1):
        cumulative += int(step.get("output_tokens", 0) or 0)
        if cumulative > budget:
            break
        if step.get("action_type") == "answer" and str(step.get("final_answer", "")).strip():
            answer_visible = True
        if evidence_step is not None and not pd.isna(evidence_step) and index >= int(evidence_step):
            evidence_visible = True
    c_budget = int(bool(metric_row["C"]) and answer_visible)
    g_budget = int(bool(metric_row["G"]) and evidence_visible)
    return c_budget, g_budget, c_budget * g_budget

budget_rows = []
for _, row in metrics_df.iterrows():
    trajectory = trajectories_by_key[(row["agent"], row["problem_id"])]
    for budget in TOKEN_BUDGETS:
        c_budget, g_budget, v_budget = budget_status(row, trajectory, budget)
        budget_rows.append({
            "agent": row["agent"], "problem_id": row["problem_id"], "budget": budget,
            "C_budget": c_budget, "G_budget": g_budget, "V_budget": v_budget,
        })
budget_frame = pd.DataFrame(budget_rows)
budget_summary = budget_frame.groupby(["agent", "budget"])[["C_budget", "G_budget", "V_budget"]].mean().reset_index()
display(budget_summary)
budget_summary.to_csv(OUTPUT_DIR / "token_budget_curves.csv", index=False)
plt.figure(figsize=(10, 6))
sns.lineplot(data=budget_summary, x="budget", y="V_budget", hue="agent", marker="o")
plt.xscale("log", base=2)
plt.ylim(0, 1)
plt.title("Verified success under visible output-token budgets")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "verified_budget_curves.png", dpi=180, bbox_inches="tight")
plt.show()


,agent,avg_billable_tokens,tokens_per_correct_solve,tokens_per_verified_solve,seconds_per_correct_solve,seconds_per_verified_solve,hidden_reasoning_share,answer_reporting_share
0,glm-5.2,1872.070000,2564.479452,3323.201183,67.400815,87.341884,0.0,0.038697
1,gpt-5-nano,3240.103333,4860.155000,5311.644809,15.185783,16.596485,0.0,0.084500
4,nemotron-3-nano,4056.350000,6577.864865,7375.181818,31.191573,34.972369,0.0,0.030075
2,minimax-m3,3432.556667,4516.521930,8801.427350,149.516641,291.365761,0.0,0.104339
3,ministral-14b,1647.160000,4533.467890,9502.846154,72.670894,152.329374,0.0,0.033110


,agent,budget,C_budget,G_budget,V_budget
0,glm-5.2,512,0.673333,0.576667,0.533333
1,glm-5.2,1024,0.693333,0.583333,0.543333
2,glm-5.2,2048,0.703333,0.606667,0.543333
3,glm-5.2,4096,0.730000,0.620000,0.563333
4,glm-5.2,8192,0.730000,0.620000,0.563333
5,gpt-5-nano,512,0.496667,0.600000,0.453333
6,gpt-5-nano,1024,0.640000,0.690000,0.586667
7,gpt-5-nano,2048,0.666667,0.726667,0.610000
8,gpt-5-nano,4096,0.666667,0.726667,0.610000
9,gpt-5-nano,8192,0.666667,0.726667,0.610000


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/2720664557.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Solution-point and post-solution-waste audit


In [11]:
solution_audit = metrics_df[[
    "agent", "problem_id", "C", "G", "solution_step", "post_waste", "pre_waste", "useful"
]].copy()
solution_audit["total_steps"] = [
    len(trajectories_by_key[(row.agent, row.problem_id)].get("steps", []))
    for row in solution_audit.itertuples()
]
solution_audit["solution_detected"] = solution_audit["solution_step"].notna()
solution_audit["solution_at_terminal_step"] = (
    solution_audit["solution_step"] == solution_audit["total_steps"]
)
solution_summary = solution_audit.groupby("agent").agg(
    solution_detection_rate=("solution_detected", "mean"),
    terminal_solution_rate=("solution_at_terminal_step", "mean"),
    mean_post_waste=("post_waste", "mean"),
    median_post_waste=("post_waste", "median"),
).reset_index()
display(solution_summary)
solution_summary.to_csv(OUTPUT_DIR / "solution_point_audit.csv", index=False)
plt.figure(figsize=(10, 5))
sns.boxplot(data=solution_audit, x="agent", y="post_waste", showfliers=False)
plt.xticks(rotation=25)
plt.title("Post-solution visible tokens")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "solution_overshoot_boxplot.png", dpi=180, bbox_inches="tight")
plt.show()


,agent,solution_detection_rate,terminal_solution_rate,mean_post_waste,median_post_waste
0,glm-5.2,0.730000,0.166667,7.310000,0.0
1,gpt-5-nano,0.666667,0.056667,98.175000,67.0
2,minimax-m3,0.760000,0.370000,65.644295,0.0
3,ministral-14b,0.363333,0.190000,2.175676,0.0
4,nemotron-3-nano,0.616667,0.066667,41.668896,0.0


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1274640131.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Step labels, critic usage, and earliest errors


In [12]:
step_rows = []
for metric_record in metrics_raw:
    agent, problem_id = metric_record["agent"], metric_record["problem_id"]
    trajectory = trajectories_by_key[(agent, problem_id)]
    details = metric_record.get("Q", {}).get("step_details", [])
    for index, detail in enumerate(details):
        step = trajectory.get("steps", [])[index] if index < len(trajectory.get("steps", [])) else {}
        step_rows.append({
            "agent": agent, "problem_id": problem_id, "C": metric_record["C_G_V"]["C_final"],
            "step_index": index + 1, "total_steps": len(details),
            "normalized_position": (index + 1) / max(len(details), 1),
            "action_type": step.get("action_type", ""), **detail,
        })
steps_df = pd.DataFrame(step_rows)
label_rates = steps_df.groupby(["agent", "label"]).size().rename("count").reset_index()
label_rates["rate"] = label_rates["count"] / label_rates.groupby("agent")["count"].transform("sum")
display(label_rates)
label_rates.to_csv(OUTPUT_DIR / "step_label_rates.csv", index=False)
label_rates.pivot(index="agent", columns="label", values="rate").fillna(0).plot(
    kind="bar", stacked=True, figsize=(11, 5), ylim=(0, 1), title="Step classification composition"
)
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "step_label_composition.png", dpi=180, bbox_inches="tight")
plt.show()

critic_audit = steps_df.groupby("agent").agg(
    steps=("step_index", "count"), critic_call_rate=("critic_used", "mean"),
    mean_prm_delta=("prm_delta", "mean"), mean_hybrid_score=("hybrid_score", "mean"),
).reset_index()
display(critic_audit)
critic_audit.to_csv(OUTPUT_DIR / "critic_step_audit.csv", index=False)

incorrect = steps_df[steps_df["C"] == 0]
earliest_error = (
    incorrect[incorrect["label"] == "regressive"]
    .groupby(["agent", "problem_id"])["normalized_position"].min().reset_index()
)
earliest_summary = earliest_error.groupby("agent")["normalized_position"].agg(["mean", "median", "count"])
display(earliest_summary)
earliest_summary.to_csv(OUTPUT_DIR / "earliest_regressive_step.csv")


,agent,label,count,rate
0,glm-5.2,neutral_useful,4,0.005814
1,glm-5.2,neutral_waste,55,0.079942
2,glm-5.2,progressive,298,0.433140
3,glm-5.2,redundant,2,0.002907
4,glm-5.2,regressive,329,0.478198
5,gpt-5-nano,neutral_useful,29,0.024786
6,gpt-5-nano,neutral_waste,273,0.233333
7,gpt-5-nano,progressive,395,0.337607
8,gpt-5-nano,redundant,85,0.072650
9,gpt-5-nano,regressive,388,0.331624


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1564850038.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,agent,steps,critic_call_rate,mean_prm_delta,mean_hybrid_score
0,glm-5.2,688,0.139535,0.023720,0.039585
1,gpt-5-nano,1170,0.090598,0.029826,0.022615
2,minimax-m3,744,0.149194,0.038578,0.040878
3,ministral-14b,640,0.348438,-0.019792,-0.143334
4,nemotron-3-nano,1141,0.070114,0.020071,-0.051121


,mean,median,count
agent,,,
glm-5.2,0.447009,0.500000,78
gpt-5-nano,0.396222,0.333333,75
minimax-m3,0.418462,0.500000,65
ministral-14b,0.490374,0.500000,187
nemotron-3-nano,0.361364,0.333333,110


## 11. PRM agreement and critic ablation


In [13]:
prm_rows = []
for agent, problem_entries in prm_scores.items():
    for problem_id, entry in problem_entries.items():
        by_model = entry.get("by_model", {})
        keys = sorted(by_model)
        max_steps = max([len(values) for values in by_model.values()] or [0])
        for index in range(max_steps):
            row = {"agent": agent, "problem_id": problem_id, "step_index": index + 1}
            for key in keys:
                row[key] = by_model[key][index] if index < len(by_model[key]) else np.nan
            prm_rows.append(row)
prm_frame = pd.DataFrame(prm_rows)
prm_model_columns = [
    column for column in prm_frame.columns
    if column not in {"agent", "problem_id", "step_index"}
]
if len(prm_model_columns) >= 2:
    agreement_rows = []
    for left, right in combinations(prm_model_columns, 2):
        pair = prm_frame[[left, right]].dropna()
        rho, p = spearmanr(pair[left], pair[right]) if len(pair) > 1 else (np.nan, np.nan)
        agreement_rows.append({
            "left": left, "right": right, "n": len(pair), "spearman_rho": rho, "p": p,
            "mean_absolute_difference": np.mean(np.abs(pair[left] - pair[right])) if len(pair) else np.nan,
        })
    prm_agreement = pd.DataFrame(agreement_rows)
    display(prm_agreement)
    prm_agreement.to_csv(OUTPUT_DIR / "prm_agreement.csv", index=False)
    paired_prm = prm_frame[prm_model_columns].dropna()
    if len(paired_prm) >= 2:
        sample = paired_prm.sample(min(5000, len(paired_prm)), random_state=RNG_SEED)
        grid = sns.pairplot(sample)
        grid.fig.savefig(OUTPUT_DIR / "prm_agreement_pairplot.png", dpi=180, bbox_inches="tight")
        plt.show()
    else:
        print("PRM pair plot skipped: fewer than two fully paired step records.")
else:
    print("PRM agreement skipped: fewer than two PRM score columns were available.")

config = manifest.get("config", {})
w_critic = float(config.get("w_critic", 0.25))
progressive_threshold = float(config.get("progressive_threshold", 0.20))
regressive_threshold = float(config.get("regressive_threshold", -0.20))

def label_without_critic(detail):
    score = float(detail.get("hybrid_score", 0.0)) - w_critic * float(detail.get("critic_signal", 0.0))
    if detail.get("error_flag") or score <= regressive_threshold:
        return "regressive"
    if detail.get("harmful_repeat"):
        return "redundant"
    if score >= progressive_threshold:
        return "progressive"
    if detail.get("useful_verification"):
        return "neutral_useful"
    return "neutral_waste"

class_value = {"progressive": 1.0, "neutral_useful": 0.5, "neutral_waste": 0.0, "redundant": -0.5, "regressive": -1.0}
critic_ablation_rows = []
for record in metrics_raw:
    details = record.get("Q", {}).get("step_details", [])
    labels = [label_without_critic(detail) for detail in details]
    signed = np.mean([class_value[label] for label in labels]) if labels else -1.0
    critic_ablation_rows.append({
        "agent": record["agent"], "problem_id": record["problem_id"],
        "Q_with_critic": record["Q"]["Q_step"], "Q_without_critic": np.clip((signed + 1) / 2, 0, 1),
    })
critic_ablation = pd.DataFrame(critic_ablation_rows)
critic_ablation_summary = critic_ablation.groupby("agent")[["Q_with_critic", "Q_without_critic"]].mean()
display(critic_ablation_summary)
critic_ablation_summary.to_csv(OUTPUT_DIR / "critic_ablation.csv")


,left,right,n,spearman_rho,p,mean_absolute_difference
0,math_shepherd_mistral_7b,qwen25_math_prm_7b,4383,0.378708,1.699232e-149,0.33344


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1423903334.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Q_with_critic,Q_without_critic
agent,,
glm-5.2,0.486250,0.481250
gpt-5-nano,0.470536,0.435292
minimax-m3,0.448476,0.443375
ministral-14b,0.164302,0.160444
nemotron-3-nano,0.416123,0.409528


## 12. Redundancy threshold sensitivity


In [14]:
redundancy_rows = []
for threshold in REDUNDANCY_THRESHOLDS:
    for agent, group in steps_df.groupby("agent"):
        repeated = (
            group["exact_repetition"].astype(bool)
            | (group["max_similarity"].fillna(0) >= threshold)
            | group["algebraic_restatement"].astype(bool)
        )
        harmful = (
            (group["action_type"] != "answer")
            & repeated
            & ~group["useful_verification"].astype(bool)
            & (group["tool_gain"].fillna(0) == 0)
        )
        redundancy_rows.append({
            "threshold": threshold, "agent": agent,
            "harmful_redundancy": harmful.mean(),
            "raw_semantic_repetition": (group["max_similarity"].fillna(0) >= threshold).mean(),
        })
redundancy_sensitivity = pd.DataFrame(redundancy_rows)
display(redundancy_sensitivity)
redundancy_sensitivity.to_csv(OUTPUT_DIR / "redundancy_threshold_sensitivity.csv", index=False)
plt.figure(figsize=(10, 6))
sns.lineplot(data=redundancy_sensitivity, x="threshold", y="harmful_redundancy", hue="agent", marker="o")
plt.ylim(0, 1)
plt.title("Harmful redundancy sensitivity")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "redundancy_sensitivity.png", dpi=180, bbox_inches="tight")
plt.show()

redundancy_outcome = metrics_df.groupby("agent").apply(
    lambda group: pd.Series({
        "harmful_when_correct": group.loc[group["C"] == 1, "R_harmful"].mean(),
        "harmful_when_incorrect": group.loc[group["C"] == 0, "R_harmful"].mean(),
        "useful_verification_when_correct": group.loc[group["C"] == 1, "R_useful"].mean(),
        "useful_verification_when_incorrect": group.loc[group["C"] == 0, "R_useful"].mean(),
    }),
)
display(redundancy_outcome)
redundancy_outcome.to_csv(OUTPUT_DIR / "redundancy_outcome_association.csv")


,threshold,agent,harmful_redundancy,raw_semantic_repetition
0,0.70,glm-5.2,0.027616,0.095930
1,0.70,gpt-5-nano,0.218803,0.290598
2,0.70,minimax-m3,0.020161,0.146505
3,0.70,ministral-14b,0.031250,0.062500
4,0.70,nemotron-3-nano,0.236635,0.309378
5,0.75,glm-5.2,0.026163,0.079942
6,0.75,gpt-5-nano,0.200855,0.244444
7,0.75,minimax-m3,0.016129,0.094086
8,0.75,ministral-14b,0.031250,0.057813
9,0.75,nemotron-3-nano,0.230500,0.284838


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1327283011.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1327283011.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  redundancy_outcome = metrics_df.groupby("agent").apply(


,harmful_when_correct,harmful_when_incorrect,useful_verification_when_correct,useful_verification_when_incorrect
agent,,,,
glm-5.2,0.005708,0.026955,0.053577,0.032922
gpt-5-nano,0.162833,0.067917,0.090250,0.057500
minimax-m3,0.004532,0.002857,0.089254,0.129762
ministral-14b,0.018043,0.007843,0.037920,0.001783
nemotron-3-nano,0.189279,0.198684,0.071532,0.054678


## 13. Latency distributions and reliability-adjusted performance


In [15]:
latency_rows = []
for agent, group in metrics_df.groupby("agent"):
    row = {"agent": agent}
    for column in ("latency", "service", "pace", "retry_wait"):
        values = group[column].dropna()
        for percentile in (50, 90, 95, 99):
            row[f"{column}_p{percentile}"] = np.percentile(values, percentile) if len(values) else np.nan
    latency_rows.append(row)
latency_percentiles = pd.DataFrame(latency_rows)
display(latency_percentiles)
latency_percentiles.to_csv(OUTPUT_DIR / "latency_percentiles.csv", index=False)

plt.figure(figsize=(10, 6))
for agent, group in metrics_df.groupby("agent"):
    values = np.sort(group["latency"].dropna().to_numpy())
    if len(values):
        plt.step(values, np.arange(1, len(values) + 1) / len(values), where="post", label=agent)
plt.xlabel("Trajectory latency (seconds)")
plt.ylabel("Empirical CDF")
plt.legend()
plt.title("Trajectory latency distributions")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latency_ecdf.png", dpi=180, bbox_inches="tight")
plt.show()

reliability_rows = []
for agent, group in metrics_df.groupby("agent"):
    completed = group["completed"]
    reliability_rows.append({
        "agent": agent,
        "availability": completed.mean(),
        "conditional_C": group.loc[completed, "C"].mean(),
        "conditional_V": group.loc[completed, "V"].mean(),
        "operational_C": group["C"].mean(),
        "operational_V": group["V"].mean(),
        "infrastructure_failure_rate": group["infrastructure_failure"].mean(),
    })
reliability = pd.DataFrame(reliability_rows).sort_values("operational_V", ascending=False)
display(reliability)
reliability.to_csv(OUTPUT_DIR / "reliability_adjusted_performance.csv", index=False)


,agent,latency_p50,latency_p90,latency_p95,latency_p99,service_p50,service_p90,service_p95,service_p99,pace_p50,pace_p90,pace_p95,pace_p99,retry_wait_p50,retry_wait_p90,retry_wait_p95,retry_wait_p99
0,glm-5.2,47.352947,80.647295,125.939167,170.976903,11.008299,58.223932,83.757088,140.990438,17.426588,46.240559,48.019093,72.059796,0.0,0.000000,0.000000,60.674407
1,gpt-5-nano,9.258505,14.857689,23.243308,50.212561,6.955124,12.310750,16.211800,25.280811,0.120520,0.345187,0.382476,0.652009,0.0,0.000000,0.000000,0.000000
2,minimax-m3,65.482515,218.894468,341.831472,928.403972,34.848877,154.949816,217.795654,395.550983,11.700032,29.633968,33.857844,50.564972,0.0,60.631465,62.045567,64.073216
3,ministral-14b,13.190003,53.174062,93.078133,150.558960,7.469024,49.122112,87.210028,143.350046,5.239533,8.812759,9.384741,13.536483,0.0,0.000000,0.000000,2.650685
4,nemotron-3-nano,13.370535,35.685059,52.056298,106.256120,3.445526,23.760823,40.822068,90.916358,7.591927,10.510529,11.185186,12.304407,0.0,0.000000,0.000000,21.459740


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/2205387020.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,agent,availability,conditional_C,conditional_V,operational_C,operational_V,infrastructure_failure_rate
1,gpt-5-nano,0.933333,0.714286,0.653571,0.666667,0.610000,0.030000
0,glm-5.2,1.000000,0.730000,0.563333,0.730000,0.563333,0.000000
4,nemotron-3-nano,0.996667,0.618729,0.551839,0.616667,0.550000,0.000000
2,minimax-m3,0.993333,0.765101,0.392617,0.760000,0.390000,0.003333
3,ministral-14b,0.986667,0.368243,0.175676,0.363333,0.173333,0.013333


## 14. Pareto frontier and bootstrap rank stability


In [16]:
pareto = metrics_df.groupby("agent").agg(
    V=("V", "mean"), C=("C", "mean"), Q=("Q", "mean"),
    R_harmful=("R_harmful", "mean"), avg_billable=("billable", "mean"),
    avg_latency=("latency", "mean"),
).reset_index()

def cost_success_frontier(frame):
    keep = []
    for index, row in frame.iterrows():
        dominated = any(
            other["avg_billable"] <= row["avg_billable"]
            and other["V"] >= row["V"]
            and (other["avg_billable"] < row["avg_billable"] or other["V"] > row["V"])
            for other_index, other in frame.iterrows() if other_index != index
        )
        keep.append(not dominated)
    return np.array(keep)

pareto["cost_V_pareto"] = cost_success_frontier(pareto)
display(pareto.sort_values("V", ascending=False))
pareto.to_csv(OUTPUT_DIR / "pareto_summary.csv", index=False)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=pareto, x="avg_billable", y="V", hue="agent", style="cost_V_pareto", s=130)
for row in pareto.itertuples():
    plt.annotate(row.agent, (row.avg_billable, row.V), xytext=(5, 5), textcoords="offset points", fontsize=8)
plt.title("Cost versus verified-success Pareto frontier")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cost_verified_pareto.png", dpi=180, bbox_inches="tight")
plt.show()

v_matrix = metrics_df.pivot_table(index="problem_id", columns="agent", values="V", aggfunc="mean")
agents = list(v_matrix.columns)
values = v_matrix.to_numpy(dtype=float)
rng = np.random.default_rng(RNG_SEED)
sampled_indices = rng.integers(0, len(values), size=(BOOTSTRAP_SAMPLES, len(values)))
sampled_means = np.nanmean(values[sampled_indices], axis=1)
ordering = np.argsort(-sampled_means, axis=1)
ranks = np.empty_like(sampled_means, dtype=float)
rows = np.arange(BOOTSTRAP_SAMPLES)[:, None]
ranks[rows, ordering] = np.arange(1, len(agents) + 1)[None, :]
rank_frame = pd.DataFrame(ranks, columns=agents).rename_axis("sample").reset_index().melt(
    id_vars="sample", var_name="agent", value_name="rank"
)
rank_stability = rank_frame.groupby("agent").agg(
    mean_rank=("rank", "mean"), median_rank=("rank", "median"),
    probability_rank_1=("rank", lambda values: np.mean(values == 1)),
).sort_values("mean_rank")
display(rank_stability)
rank_stability.to_csv(OUTPUT_DIR / "bootstrap_rank_stability.csv")


,agent,V,C,Q,R_harmful,avg_billable,avg_latency,cost_V_pareto
1,gpt-5-nano,0.610000,0.666667,0.470536,0.135714,3240.103333,10.123856,True
0,glm-5.2,0.563333,0.730000,0.486250,0.011444,1872.070000,49.202595,True
4,nemotron-3-nano,0.550000,0.616667,0.416123,0.192865,4056.350000,19.234803,False
2,minimax-m3,0.390000,0.760000,0.448476,0.004139,3432.556667,113.632647,False
3,ministral-14b,0.173333,0.363333,0.164302,0.011599,1647.160000,26.403758,True


/var/folders/8x/l0z1dx8518d6pg_4nnhz45pc0000gn/T/ipykernel_66768/1077367588.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean_rank,median_rank,probability_rank_1
agent,,,
gpt-5-nano,1.0926,1.0,0.9154
glm-5.2,2.2350,2.0,0.0682
nemotron-3-nano,2.6724,3.0,0.0164
minimax-m3,4.0000,4.0,0.0000
ministral-14b,5.0000,5.0,0.0000


## 15. Export all analysis outputs


In [17]:
analysis_manifest = {
    "source_run": manifest.get("run_name"),
    "source_manifest_sha256": hashlib.sha256(
        json.dumps(manifest, sort_keys=True).encode()
    ).hexdigest(),
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "token_budgets": TOKEN_BUDGETS,
    "redundancy_thresholds": REDUNDANCY_THRESHOLDS,
    "operational_agents": sorted(set(operational_df["agent"])),
    "trace_comparable_agents": trace_agents,
    "trace_excluded_agents": sorted(TRACE_EXCLUDED_AGENTS),
    "notes": {
        "bundle_rule": "The merged paper artifact is already deduplicated by (agent, problem_id).",
        "budget_curve": "Visible API output-token prefix budget; hidden reasoning availability varies by provider.",
        "post_solution": "Current solution point may be terminal for trajectories without traceable tool evidence.",
        "paired_tests": "Paired by problem_id; McNemar p-values use Holm correction.",
        "gpt_oss": "Retained operationally; excluded from trace-derived analyses due to unmapped provider reasoning.",
    },
}
(OUTPUT_DIR / "analysis_manifest.json").write_text(
    json.dumps(analysis_manifest, indent=2), encoding="utf-8"
)
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
print("Advanced analysis directory:", OUTPUT_DIR)
print("Downloadable archive:", archive)
try:
    from IPython.display import FileLink, display as ipy_display
    ipy_display(FileLink(str(archive)))
except Exception:
    pass


Advanced analysis directory: /path/to/your/project/strive_final_advanced_analysis
Downloadable archive: /path/to/your/project/strive_final_advanced_analysis.zip


/path/to/your/project/strive_final_advanced_analysis.zip